# Lista de Exercícios - Isolation Forest

Resolução dos 8 exercícios usando `IsolationForest` do scikit-learn.

Os arquivos `sensores_ambientais.csv` e `vendas_ecommerce.csv` (exercícios 7 e 8) precisam estar
na mesma pasta que esse notebook, senão o `pd.read_csv` não acha.


### Setup

In [ ]:
%pip install -q pandas scikit-learn matplotlib


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

def carregar_csv(nome_arquivo):
    import os
    if not os.path.exists(nome_arquivo):
        raise FileNotFoundError(f"cade o arquivo {nome_arquivo}? bota ele na mesma pasta do notebook")
    return pd.read_csv(nome_arquivo)

np.random.seed(42)


## Exercício 1 - Vendas de uma mercearia

Cada valor é a qtd de produtos vendidos em um dia. Uma característica só (quantidade), então é o
caso mais simples pra treinar o modelo.


In [ ]:
vendas = [
    [80], [85], [90], [88],
    [92], [87], [95], [89],
    [91], [400]
]

df_vendas = pd.DataFrame(vendas, columns=["quantidade_vendida"])

# 1 candidato a cada 10 obs = contamination 0.1
modelo_vendas = IsolationForest(contamination=0.1, random_state=42)
df_vendas["rotulo"] = modelo_vendas.fit_predict(df_vendas[["quantidade_vendida"]])
df_vendas["pontuacao_anomalia"] = modelo_vendas.decision_function(df_vendas[["quantidade_vendida"]])

df_vendas


In [ ]:
candidatos = df_vendas[df_vendas["rotulo"] == -1]
candidatos


O modelo pegou o 400 mesmo, que era o esperado, já que os outros dias ficam todos entre 80 e 95.
400 é praticamente 4x a média dos outros, então nem precisa de muita análise pra ver que destoa.

Sobre o motivo, dá pra pensar em umas 3 hipóteses: pode ter tido uma promoção naquele dia (aumento
de demanda real), pode ter sido uma encomenda grande de algum cliente (tipo um mercado menor
comprando em lote), ou simplesmente alguém digitou "400" no lugar de "40" quando lançou a venda.
O modelo não sabe diferenciar essas situações, ele só aponta que o valor é estatisticamente
estranho.

**O valor 400 deve ser removido automaticamente?** Não acho que seja uma boa ideia. O Isolation
Forest serve pra levantar a suspeita, mas quem decide se apaga, corrige ou mantém tem que ser uma
pessoa olhando o caso (conferir a nota fiscal daquele dia, por exemplo). Se for promoção ou
encomenda legítima, remover o dado ia esconder uma informação real do negócio.


## Exercício 2 - Latência de uma conexão de internet

Inventei 12 medições de tempo de resposta em ms. A maioria fica entre 20 e 70 ms, e coloquei duas
bem acima de 200 ms de propósito.


In [ ]:
latencias = [
    [25], [30], [45], [60],
    [35], [50], [65], [28],
    [55], [40], [220], [310]
]

df_latencia = pd.DataFrame(latencias, columns=["latencia_ms"])

modelo_latencia = IsolationForest(contamination=2/len(df_latencia), random_state=42)
df_latencia["rotulo"] = modelo_latencia.fit_predict(df_latencia[["latencia_ms"]])

df_latencia


In [ ]:
df_latencia[df_latencia["rotulo"] == -1]


Como esperado, os dois valores acima de 200 ms (220 e 310) foram os únicos marcados como
candidatos. Faz sentido, porque o resto das medições fica todo numa faixa de 20-65 ms, então esses
dois pontos ficam bem longe do grupo.

**Uma latência alta sempre representa uma falha?** Acho que não, na real. Uma medição isolada de
latência alta pode ser só um pico momentâneo (rede congestionada por um segundo, servidor
processando algo mais pesado naquele instante etc). Pra falar que é falha de verdade eu ia querer
ver várias medições altas seguidas, não só um ponto fora da curva.


## Exercício 3 - Consumo de combustível

Agora com duas colunas: distância percorrida e litros consumidos.


In [ ]:
consumo = [
    [10, 1.0], [20, 1.8], [30, 2.6],
    [40, 3.5], [50, 4.3], [60, 5.1],
    [70, 6.0], [80, 7.0], [90, 8.0],
    [100, 20.0]
]

df_consumo = pd.DataFrame(consumo, columns=["distancia_km", "litros_consumidos"])

modelo_consumo = IsolationForest(contamination=0.1, random_state=42)
df_consumo["rotulo"] = modelo_consumo.fit_predict(df_consumo[["distancia_km", "litros_consumidos"]])

df_consumo


In [ ]:
plt.figure(figsize=(6, 5))
normais = df_consumo[df_consumo["rotulo"] == 1]
anomalos = df_consumo[df_consumo["rotulo"] == -1]

plt.scatter(normais["distancia_km"], normais["litros_consumidos"], c="steelblue", label="normal")
plt.scatter(anomalos["distancia_km"], anomalos["litros_consumidos"], c="crimson", marker="x", s=120, label="candidato")

plt.xlabel("distancia (km)")
plt.ylabel("litros")
plt.legend()
plt.tight_layout()
plt.show()


A linha de 100 km / 20 litros foi a marcada. Olhando o padrão dos outros carros, pra 100 km o
esperado seria uns 8-9 litros (uns 11-12 km/l), mas esse registro consumiu praticamente o dobro
disso.

O interessante desse exercício é que, se eu olhar só a coluna distância, 100 km não é nada absurdo
(é só o maior valor da lista, meio que uma continuação natural). E se eu olhar só litros, 20 também
não parece tão fora assim comparado a uma escala maior. O problema é a combinação das duas coisas:
gastar 20 litros pra rodar só 100 km é uma eficiência muito ruim (~5 km/l) pertinho de veículos que
rendem o dobro disso. Por isso não dá pra analisar as colunas separadas, tem que olhar a relação
entre elas.


## Exercício 4 - Viagens de ônibus urbano

Duas colunas de novo: passageiros e atraso em minutos.


In [ ]:
viagens = [
    [32, 3], [45, 5], [50, 4],
    [60, 6], [55, 5], [70, 7],
    [65, 6], [80, 8], [75, 7],
    [10, 45]
]

df_viagens = pd.DataFrame(viagens, columns=["passageiros", "atraso_min"])

modelo_viagens = IsolationForest(contamination=0.1, random_state=42)
df_viagens["rotulo"] = modelo_viagens.fit_predict(df_viagens[["passageiros", "atraso_min"]])

df_viagens


In [ ]:
plt.figure(figsize=(6, 5))
normais = df_viagens[df_viagens["rotulo"] == 1]
anomalos = df_viagens[df_viagens["rotulo"] == -1]

plt.scatter(normais["passageiros"], normais["atraso_min"], c="steelblue", label="normal")
plt.scatter(anomalos["passageiros"], anomalos["atraso_min"], c="crimson", marker="x", s=120, label="candidato")

plt.xlabel("passageiros")
plt.ylabel("atraso (min)")
plt.legend()
plt.tight_layout()
plt.show()


A viagem com 10 passageiros e 45 min de atraso ficou marcada, e é bem óbvio olhando o gráfico -
ela fica sozinha lá no canto, longe de todo mundo. Todas as outras viagens têm entre 32 e 80
passageiros com atraso de no máximo 8 minutos, então essa combinação de poucos passageiros e muito
atraso quebra o padrão dos dois jeitos ao mesmo tempo.

Pensando em causas possíveis: pode ter tido um engarrafamento feio na rota, um acidente no
caminho, o ônibus pode ter quebrado (problema mecânico obrigando parar), ou até um desvio de
percurso. Os poucos passageiros também podem ser consequência disso - se a viagem atrasou muito,
talvez parte do pessoal tenha desistido de esperar e pego outro transporte.


## Exercício 5 - Regra fixa vs Isolation Forest

Aqui a ideia é comparar duas abordagens pro mesmo problema: uma regra de limite fixo (tipo "se for
menor que X ou maior que Y, marca") contra o Isolation Forest.


In [ ]:
consumo_agua = [
    [48], [52], [50], [55],
    [49], [53], [51], [54],
    [56], [12], [180]
]

df_agua = pd.DataFrame(consumo_agua, columns=["consumo_litros"])

# regra fixa: qualquer coisa fora de 40-60 é suspeito
LIMITE_INFERIOR, LIMITE_SUPERIOR = 40, 60
df_agua["candidato_regra"] = ~df_agua["consumo_litros"].between(LIMITE_INFERIOR, LIMITE_SUPERIOR)

modelo_agua = IsolationForest(contamination=0.2, random_state=42)
df_agua["rotulo_iforest"] = modelo_agua.fit_predict(df_agua[["consumo_litros"]])
df_agua["candidato_iforest"] = df_agua["rotulo_iforest"] == -1

df_agua


In [ ]:
iguais = (df_agua["candidato_regra"] == df_agua["candidato_iforest"]).all()
print("os dois métodos concordaram em todos os pontos?", iguais)


Nesse caso os dois bateram: tanto a regra quanto o modelo marcaram só o 12 e o 180, que são os
valores óbvios de qualquer jeito que se olhe. Mas isso não quer dizer que a regra é tão boa quanto
o modelo - ela só funcionou porque eu escolhi 40 e 60 depois de já ter visto os dados, o que na
prática é meio "trapaça".

O problema da regra fixa é justamente esse: alguém tem que definir esses limites de antemão, e sem
saber como os dados normais se comportam isso vira meio chute. Além disso a regra não se adapta se
o padrão mudar com o tempo (por exemplo, se mais gente for morar na casa e o consumo normal subir
pra faixa dos 60-70), e ela não escala bem quando tem mais de uma variável (tipo nos exercícios 3 e
4, onde o problema é a combinação de duas colunas, não uma regra simples em cada uma).

O Isolation Forest não precisa que ninguém defina esse limite, ele aprende olhando a distribuição
dos próprios dados.

**É melhor definir manualmente um limite ou deixar o modelo aprender sozinho?** Acho que depende.
Se existe um limite físico bem conhecido (tipo um sensor que não pode passar de tal valor), regra
fixa resolve e é mais simples de explicar pra qualquer pessoa. Mas quando o "normal" é uma coisa
mais complexa, com várias variáveis ou que muda com o tempo, acho que vale mais a pena deixar o
modelo aprender do que ficar chutando limite.


## Exercício 6 - Problema completo (escolhi velocidade de veículos)

Dataset inventado: velocidade de carros medida por um radar (km/h). Coloquei 17 observações, sendo
15 "normais" e 2 bem fora do padrão.


In [ ]:
velocidades = [
    [42], [45], [38], [50], [47],
    [44], [39], [48], [46], [41],
    [43], [49], [45], [40], [47],
    [120],  # bem acima do limite da via
    [5],    # quase parado
]

df_velocidade = pd.DataFrame(velocidades, columns=["velocidade_kmh"])

modelo_velocidade = IsolationForest(contamination=2/len(df_velocidade), random_state=42)
df_velocidade["rotulo"] = modelo_velocidade.fit_predict(df_velocidade[["velocidade_kmh"]])

df_velocidade


In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(df_velocidade.index, df_velocidade["velocidade_kmh"],
            c=df_velocidade["rotulo"].map({1: "steelblue", -1: "crimson"}), s=80)
plt.axhspan(38, 50, color="steelblue", alpha=0.08)
plt.xlabel("ordem da medição")
plt.ylabel("velocidade (km/h)")
plt.tight_layout()
plt.show()


Os dois candidatos foram exatamente os que eu adicionei de propósito: 120 km/h e 5 km/h.

O 120 provavelmente é um evento real (alguém dirigindo rápido demais), não parece erro de sensor -
seria um caso pra investigar por excesso de velocidade mesmo. Já o 5 km/h pode ser trânsito parado,
uma pane no carro, ou alguém fazendo manobra bem devagar. Nenhum dos dois parece exatamente um
"erro de medição", os dois são mais tipo situações incomuns que aconteceram de verdade.

Sobre o critério: basicamente foi a distância em relação ao resto do grupo. Quase todas as
velocidades ficaram entre 38 e 50 km/h, um bloco bem apertado, e os dois pontos extras ficam bem
longe disso pros dois lados. É mais fácil "isolar" esses pontos com poucas divisões, que é
basicamente como o algoritmo funciona por trás.


## Exercício 7 - Sensores ambientais (CSV)

Agora usando o arquivo `sensores_ambientais.csv`, com temperatura e umidade de vários ambientes.


In [ ]:
dados_sensores = carregar_csv("sensores_ambientais.csv")
dados_sensores.head()


In [ ]:
caracteristicas_sensores = dados_sensores[["temperatura", "umidade"]]

modelo_sensores = IsolationForest(contamination=0.1, random_state=42)
dados_sensores["rotulo"] = modelo_sensores.fit_predict(caracteristicas_sensores)

leituras_candidatas = dados_sensores[dados_sensores["rotulo"] == -1]
print(leituras_candidatas["id_leitura"].tolist())
leituras_candidatas


In [ ]:
plt.figure(figsize=(6, 5))
normais = dados_sensores[dados_sensores["rotulo"] == 1]
anomalos = dados_sensores[dados_sensores["rotulo"] == -1]

plt.scatter(normais["temperatura"], normais["umidade"], c="steelblue", label="normal")
plt.scatter(anomalos["temperatura"], anomalos["umidade"], c="crimson", marker="x", s=120, label="candidato")

for _, linha in anomalos.iterrows():
    plt.annotate(linha["id_leitura"], (linha["temperatura"], linha["umidade"]), xytext=(6, 6), textcoords="offset points")

plt.xlabel("temperatura")
plt.ylabel("umidade")
plt.legend()
plt.tight_layout()
plt.show()


As leituras L011 e L017 foram sinalizadas. A maioria das leituras fica num intervalo bem
consistente (temperatura em torno de 20-24°C, umidade entre 47-61%), típico de um ambiente
fechado/climatizado. Essas duas fogem disso: uma delas tem temperatura bem mais alta com umidade
bem mais baixa, e a outra é o oposto, temperatura baixa e umidade bem alta.

Pode ser janela ou porta aberta em algum desses momentos, um ambiente diferente sendo medido por
engano (área externa, perto de alguma fonte de calor), ou o sensor deu uma pane momentânea. Sem
mais contexto não dá pra saber qual das três é, mas os dois casos merecem uma olhada.


## Exercício 8 - Vendas de e-commerce (CSV)

Usando `vendas_ecommerce.csv`. O enunciado pede pra não usar `id_venda` como feature, faz sentido
porque é só um identificador, não carrega informação sobre o comportamento da venda.


In [ ]:
dados_vendas = carregar_csv("vendas_ecommerce.csv")
dados_vendas.head()


In [ ]:
caracteristicas_vendas = dados_vendas[["valor_total", "quantidade_itens", "desconto_percentual"]]

modelo_vendas_ecom = IsolationForest(contamination=0.1, random_state=42)
dados_vendas["rotulo"] = modelo_vendas_ecom.fit_predict(caracteristicas_vendas)

vendas_candidatas = dados_vendas[dados_vendas["rotulo"] == -1]
vendas_candidatas


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

normais = dados_vendas[dados_vendas["rotulo"] == 1]
anomalos = dados_vendas[dados_vendas["rotulo"] == -1]

axes[0].scatter(normais["valor_total"], normais["quantidade_itens"], c="steelblue", label="normal")
axes[0].scatter(anomalos["valor_total"], anomalos["quantidade_itens"], c="crimson", marker="x", s=120, label="candidato")
axes[0].set_xlabel("valor total")
axes[0].set_ylabel("qtd itens")
axes[0].legend()

axes[1].scatter(normais["valor_total"], normais["desconto_percentual"], c="steelblue", label="normal")
axes[1].scatter(anomalos["valor_total"], anomalos["desconto_percentual"], c="crimson", marker="x", s=120, label="candidato")
axes[1].set_xlabel("valor total")
axes[1].set_ylabel("desconto (%)")
axes[1].legend()

plt.tight_layout()
plt.show()


V011 e V020 foram as duas vendas marcadas. Nos dois casos o valor total é bem mais alto que o resto
(5800 e 4200, contra a maioria que fica entre 70 e 420), com poucos itens comprados e um desconto
percentual gigante (90% e 85%). No resto da base o desconto costuma acompanhar mais ou menos o
tamanho da compra e fica numa faixa de 0-20%, então esses dois pontos destoam bastante.

Dá pra pensar em algumas explicações: pode ser erro de digitação no valor (um zero a mais sem
querer), pode ser uma promoção muito agressiva mesmo (tipo liquidação de estoque parado), ou pode
ser algo que merece investigar melhor, tipo abuso de cupom de desconto. Sem saber o contexto da
loja (se tinha campanha ativa naquele dia, etc) não dá pra cravar qual é o motivo certo, mas de
qualquer forma vale a pena alguém conferir esses dois casos antes de assumir que é erro.
